# Fine-Tune DeepSeek R1 for Lab Test Analysis (RTX 2060 / 6GB VRAM)

This notebook demonstrates how to fine-tune the `unsloth/DeepSeek-R1-Distill-Llama-8B` model on a custom medical lab test dataset using Unsloth and LoRA, optimized for RTX 2060 GPUs with 6GB VRAM.

In [ ]:
%%capture
# Install Unsloth with latest optimizations for local PC
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo
!pip install datasets wandb trl peft transformers bitsandbytes python-dotenv

## 1. Authentication
Log in to Hugging Face and Weights & Biases (optional but recommended for tracking).

In [1]:
from huggingface_hub import login
import wandb
import os
from dotenv import load_dotenv

# Load secrets from .env (local PC approach)
dotenv_path = os.path.join(os.getcwd(), '.env')
if not os.path.exists(dotenv_path):
    if os.path.exists('../.env'):
        dotenv_path = '../.env'

print(f"Loading .env from: {os.path.abspath(dotenv_path)}")
loaded = load_dotenv(dotenv_path=dotenv_path)
if not loaded:
    print("⚠️  .env file NOT loaded. Checking environment variables...")

# Hugging Face Login
hf_token = os.environ.get("HUGGINGFACE_TOKEN")
if hf_token and hf_token.startswith("hf_"):
    login(token=hf_token)
    print("✅ Hugging Face: Logged in successfully.")
else:
    print("❌ CRITICAL: HUGGINGFACE_TOKEN not found or invalid.")
    print("   Please add it to your .env file.")

# Weights & Biases Login (optional)
wb_token = os.environ.get("WANDB_API_KEY")
if wb_token:
    wandb.login(key=wb_token)
    run = wandb.init(
        project='Fine-tune-DeepSeek-R1-Lab-Tests',
        job_type='training',
        anonymous='allow'
    )
    print("✅ W&B: Tracking enabled.")
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print("ℹ️  W&B: No key found — tracking disabled.")

Loading .env from: /home/izzytn/projects/medlab/.env


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/izzytn/.netrc


✅ Hugging Face: Logged in successfully.


wandb: Currently logged in as: nguumatn (nguumatn-entrick-information-systems) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


✅ W&B: Tracking enabled.


## 1.5 GPU Check
Verify GPU availability. RTX 2060 with 6GB VRAM supports 4-bit quantization and LoRA fine-tuning.

In [2]:
import torch

if torch.cuda.is_available():
    device_name   = torch.cuda.get_device_name(0)
    total_memory  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {device_name}")
    print(f"   Total VRAM: {total_memory:.2f} GB")
    print(f"   PyTorch:    {torch.__version__}")
    print(f"   CUDA:       {torch.version.cuda}")
else:
    print("❌ No GPU detected!")
    print("   Please ensure you have a CUDA-capable GPU installed and configured.")
    import sys
    print(f"   Python: {sys.executable}")
    raise RuntimeError("GPU required. Please check your CUDA setup.")

✅ GPU: NVIDIA GeForce RTX 2060 with Max-Q Design
   Total VRAM: 6.44 GB
   PyTorch:    2.6.0+cu124
   CUDA:       12.4


## 2. Load Model & Tokenizer
Using Unsloth's `FastLanguageModel` with 4-bit quantization. Sequence length auto-reduces if needed to fit 6GB VRAM.

In [ ]:
import os
import gc
import torch
from unsloth import FastLanguageModel

MODEL_NAME     = "unsloth/DeepSeek-R1-Distill-Llama-8B"
max_seq_length = 64  # CRITICAL: 64 tokens max for 6GB VRAM with fused loss
dtype          = None
load_in_4bit   = True

print(f"🔍 GPU Status:")
print(f"   Device: {torch.cuda.get_device_name(0)}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n📡 Loading model: {MODEL_NAME}")
print(f"   Sequence length: {max_seq_length} tokens")
print(f"   Using 4-bit quantization")
print(f"   HF Token: {'✅ Found' if hf_token else '❌ NOT FOUND'}")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = max_seq_length,
        dtype          = dtype,
        load_in_4bit   = load_in_4bit,
        token          = hf_token,
        device_map     = "cuda",
    )
    print(f"✅ Model loaded successfully on GPU!")
    
except Exception as e:
    print(f"\n❌ Load failed:")
    print(f"   {type(e).__name__}: {str(e)[:200]}")
    raise RuntimeError(f"Failed to load {MODEL_NAME}")

print(f"\n✅ Model loaded: {MODEL_NAME}")
print(f"   Max sequence length: {max_seq_length} tokens")
print(f"   Params: {sum(p.numel() for p in model.parameters()):,} total")

🔍 GPU Status:
   Device: NVIDIA GeForce RTX 2060 with Max-Q Design
   CUDA Available: True
   VRAM: 6.44 GB

📡 Loading model: unsloth/DeepSeek-R1-Distill-Llama-8B
   Sequence length: 64 tokens
   Using 4-bit quantization
   HF Token: ✅ Found
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 2060 with Max-Q Design. Num GPUs = 1. Max memory: 6.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


## 3. Configure LoRA Adapters
LoRA (rank=8) trains only 0.4% of parameters, enabling efficient fine-tuning on limited VRAM.

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 8,    # REDUCED LoRA rank for 4GB VRAM (was 16)
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha               = 16,
    lora_dropout             = 0,    # 0 is recommended by Unsloth
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",  # Saves VRAM for long contexts
    random_state             = 3407,
    use_rslora               = False,
    loftq_config             = None,
)

print(f"✅ LoRA adapters attached.")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

✅ LoRA adapters attached.
   Trainable params: 20,971,520


## 4. Prepare the Dataset
Load the local JSONL conversational dataset and format it for the causal language model.

In [ ]:
from datasets import load_dataset

# Load the local dataset
DATASET_FILE = "fine_tuning_lab_tests.jsonl"
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"✅ Dataset loaded: {len(dataset)} examples")
print("   Sample keys:", dataset.column_names)

# DeepSeek-R1 specific prompt format (uses <think> tags for chain-of-thought)
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
{system}

### Question:
{user}

### Response:
<think>
{thought}</think>
{response}"""

def formatting_prompts_func(examples):
    texts = []
    for messages in examples['messages']:
        system    = messages[0]['content']
        user      = messages[1]['content']
        assistant = messages[2]['content']

        thought_process = (
            "Analyzing the lab result against the reference range... "
            "Determining if the value is high, low, or normal... "
            "Formulating clinical recommendations based on standard medical guidelines."
        )

        text = prompt_style.format(
            system   = system,
            user     = user,
            thought  = thought_process,
            response = assistant
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# CRITICAL: Truncate examples to fit max_seq_length=64
def truncate_examples(examples):
    truncated_texts = []
    for text in examples['text']:
        # Tokenize to count tokens
        tokens = tokenizer(text, truncation=False, return_attention_mask=False)['input_ids']
        if len(tokens) > 64:
            # Truncate to 64 tokens and re-decode
            truncated_ids = tokens[:64]
            truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=False)
            truncated_texts.append(truncated_text)
        else:
            truncated_texts.append(text)
    return {"text": truncated_texts}

dataset = dataset.map(truncate_examples, batched=True)

print(f"✅ Dataset formatted and truncated to max_seq_length=64")
print(f"   Sample:")
print(dataset[0]['text'][:200], "...")

✅ Dataset loaded: 100 examples
   Sample keys: ['messages']
✅ Dataset formatted. Sample:
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

# ...


## 5. Training Setup
Optimized for RTX 2060 (6GB VRAM) with aggressive memory reduction.

| Setting | Value | Reason |
|---|---|---|
| `max_seq_length` | 64 | Critical limit for fused loss on 6GB VRAM |
| `per_device_train_batch_size` | 1 | Minimal batch size |
| `gradient_accumulation_steps` | 1 | No accumulation (effective batch = 1) |
| `LoRA rank` | 8 | Memory-efficient adapter |
| `max_steps` | 60 | Full training demo |
| `optim` | adamw_8bit | Saves VRAM |
| `device_map` | cuda | GPU-only |
| `dataloader_pin_memory` | False | Reduce memory pressure |
| `gradient_checkpointing` | Enabled | Trades compute for memory |
| Dataset | Pre-truncated | All examples ≤64 tokens |

**Warning:** RTX 2060 (6GB VRAM) is at the absolute limit. Sequence length cannot be increased further without additional GPU memory.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print(f"🚀 Training Setup:")
print(f"   Sequence length: {max_seq_length}")
print(f"   Batch size: 1")
print(f"   Gradient accumulation: 1 (effective batch = 1)")
print(f"   Gradient Checkpointing: ENABLED")

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 1,
    args = TrainingArguments(
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 1,
        warmup_steps                 = 2,
        max_steps                    = 60,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "linear",
        seed                         = 3407,
        output_dir                   = "outputs",
        report_to                    = "none",
        save_strategy                = "no",
        remove_unused_columns        = True,
        dataloader_pin_memory        = False,
        disable_tqdm                 = False,
    ),
)

print("\n🔄 Starting training...")
print(f"   Using seq_length={max_seq_length} (small enough for fused loss on 6GB)")
trainer_stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Runtime: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"   Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 6. Inference / Testing
Test the newly fine-tuned model.

In [ ]:
import json
import re
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ── 1. Define the test ───────────────────────────────────────────
test_system = "You are a medical laboratory assistant. Analyze lab results, identify abnormalities based on reference ranges, and provide brief explanations for healthcare professionals."
test_user   = "Glucose (Fasting). Result: 155 mg/dL. Ref Range: 70-99 mg/dL."

inference_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
{test_system}

### Question:
{test_user}

### Response:
<think>"""

# ── 2. Run inference ─────────────────────────────────────────────
inputs  = tokenizer([inference_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids          = inputs.input_ids,
    attention_mask     = inputs.attention_mask,
    max_new_tokens     = 300,
    use_cache          = True,
    do_sample          = True,
    temperature        = 0.7,
    repetition_penalty = 1.3,
    eos_token_id       = tokenizer.eos_token_id,
)
raw_output    = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
response_text = raw_output.split("### Response:")[-1].strip()

# ── 3. Parse think + answer ──────────────────────────────────────
think_match = re.search(r"<think>(.*?)</think>", response_text, re.DOTALL)
reasoning   = think_match.group(1).strip() if think_match else ""
answer      = re.sub(r"<think>.*?</think>", "", response_text, flags=re.DOTALL).strip()

# ── 4. Extract structured fields ─────────────────────────────────
severity_match = re.match(r"^(Critical|High|Normal|Low|Borderline)[:\s]*([^.]*\.?)", answer, re.IGNORECASE)
severity       = severity_match.group(1).strip() if severity_match else "Unknown"
summary        = severity_match.group(2).strip() if severity_match else answer[:120]

reco_sentences = [
    s.strip() for s in re.split(r"(?<=[.!?])\s+", answer)
    if any(k in s.lower() for k in ["recommend", "suggest", "consider", "refer", "monitor", "prescribe", "urgent"])
]

test_name  = test_user.split(".")[0].strip()
result_val = (re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) or re.search(r"", "")).group(1) if re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) else ""
ref_range  = (re.search(r"Ref Range:\s*(.+)", test_user) or re.search(r"", "")).group(1).strip() if re.search(r"Ref Range:\s*(.+)", test_user) else ""

# ── 5. Build and print JSON ──────────────────────────────────────
result = {
    "lab_test": {
        "name":            test_name,
        "result":          result_val,
        "reference_range": ref_range,
    },
    "analysis": {
        "severity": severity,
        "summary":  summary,
        "full_clinical_response": answer,
    },
    "chain_of_thought":  reasoning,
    "recommendations":   reco_sentences,
}

print(json.dumps(result, indent=2))

## 7. Save the Model

**Option A** — Save LoRA adapters only (small, ~150MB, requires base model for inference)

**Option B** — Save as merged 16-bit model (large, ~16GB, self-contained)

**Option C** — Push directly to Hugging Face Hub

In [ ]:
NEW_MODEL_NAME = "Fine-tuned-DeepSeek-R1-Lab-Test-Assistant-LoRA"
SAVE_PATH      = NEW_MODEL_NAME

# --- Option A: Save LoRA adapters only (recommended for quick saving) ---
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA adapters saved to: {SAVE_PATH}")

# --- Option B: Save merged 16-bit model (uncomment if needed) ---
# model.save_pretrained_merged(SAVE_PATH + '-merged', tokenizer, save_method='merged_16bit')
# print(f"✅ Merged model saved to: {SAVE_PATH}-merged")

# --- Option C: Push to Hugging Face Hub (uncomment & set your username) ---
# HF_USERNAME = "your_username"
# HF_REPO     = f"{HF_USERNAME}/{NEW_MODEL_NAME}"
# model.push_to_hub(HF_REPO, token=hf_token)
# tokenizer.push_to_hub(HF_REPO, token=hf_token)
# print(f"✅ Model pushed to Hub: https://huggingface.co/{HF_REPO}")